# Module 1: PyTorch Fundamentals

Welcome to PyTorch! Before we build diffusion models, we need to be fluent in the framework that makes it all possible. By the end of this module, you'll be able to create tensors, trace gradients through computation graphs, build custom `nn.Module` classes, and implement layers, losses, and optimizers from scratch.

Here's what we'll work through together:

- **Tensors and devices** — creating data, moving it to GPU, and the `randn_like` pattern that powers diffusion noise sampling
- **Autograd** — how PyTorch builds computation graphs and computes gradients automatically
- **In-place operations** — why `add_()` can silently break your training
- **`nn.Module`** — parameters, buffers, submodules, and how to build your own layers
- **Custom layers** — implementing `Linear`, `Conv2d`, and `GroupNorm` from raw math
- **Loss functions** — MSE (the diffusion training loss), cross-entropy, and Huber loss from scratch
- **Optimizers** — what SGD and Adam actually do, and implementing Adam yourself

**Prerequisites:** Module 0 (NumPy Foundations). Comfortable with Python classes and linear algebra basics.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from typing import Optional, Tuple, List

torch.manual_seed(42)

# Device-agnostic setup
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## 1.1 — Tensor Creation, dtypes, Device Management

If you've used NumPy, PyTorch tensors will feel familiar — they're multi-dimensional arrays with the same slicing and broadcasting rules. The key difference: tensors can live on a GPU and track their own computation history for automatic differentiation.

Let's start with the most common ways to create them.

### Creation functions

| Function | What it does | Typical use |
|---|---|---|
| `torch.tensor(data)` | Create from a list or ndarray | Loading specific values |
| `torch.zeros(shape)` | All zeros | Bias init, accumulators |
| `torch.ones(shape)` | All ones | Mask initialization |
| `torch.randn(shape)` | Standard normal $\mathcal{N}(0,1)$ | Weight init, noise sampling |
| `torch.arange(start, end)` | Evenly spaced integers | Positional indices |
| `torch.linspace(start, end, steps)` | Evenly spaced floats | Noise schedules |

### The `*_like` factory functions

These deserve special attention. `torch.randn_like(x)` creates noise with the **same shape, dtype, and device** as `x` — no mismatches possible. In the diffusion forward process, this is exactly how we sample $\varepsilon$:

```python
noise = torch.randn_like(x_0)  # matches everything automatically
```

### dtypes

| dtype | Bits | When to use |
|---|---|---|
| `torch.float32` | 32 | Default for training — always use unless you have a reason not to |
| `torch.float64` | 64 | Numerical verification only — too slow for real training |
| `torch.int64` | 64 | Indices, labels, timestep integers |
| `torch.bool` | 8 | Masks (attention, dropout) |
| `torch.float16` / `torch.bfloat16` | 16 | Mixed-precision training |

In [ ]:
# --- Tensor Creation ---

# From Python data
x = torch.tensor([1.0, 2.0, 3.0])  # float32 by default
print(f"From list:   {x}, dtype={x.dtype}")

# Specific dtypes
labels = torch.tensor([0, 3, 7], dtype=torch.int64)  # class labels
mask = torch.tensor([True, False, True], dtype=torch.bool)  # attention mask
print(f"Labels:      {labels}, dtype={labels.dtype}")
print(f"Mask:        {mask}, dtype={mask.dtype}")

# Common creation functions
zeros = torch.zeros(2, 3)           # (2, 3)
noise = torch.randn(4, 3, 32, 32)  # (B, C, H, W) -- batch of images
steps = torch.arange(0, 1000)       # [0, 1, ..., 999] -- timestep indices
betas = torch.linspace(1e-4, 0.02, 1000)  # noise schedule values

print(f"\nzeros shape: {zeros.shape}")
print(f"noise shape: {noise.shape}  -- a batch of 4 RGB 32x32 images")
print(f"steps shape: {steps.shape}, first 5: {steps[:5]}")
print(f"betas shape: {betas.shape}, range: [{betas[0]:.4f}, {betas[-1]:.4f}]")

In [ ]:
# --- Device Management ---

# Create on CPU (default), then move
x_cpu = torch.randn(3, 3)  # (3, 3) on CPU
print(f"x_cpu device: {x_cpu.device}")

# Move to best available device
x_device = x_cpu.to(device)  # (3, 3) on device
print(f"x_device device: {x_device.device}")

# .to() returns a NEW tensor if device changes, same tensor if already there
x_same = x_device.to(device)
print(f"Same object after .to() to same device? {x_same is x_device}")  # True -- no copy
x_back = x_device.to('cpu')
print(f"Same object after .to('cpu')? {x_back is x_device}")  # False -- new tensor

# Create directly on device
y = torch.randn(3, 3, device=device)  # (3, 3)
print(f"\nCreated directly on device: {y.device}")

# IMPORTANT: Operations require tensors on the SAME device
try:
    _ = x_cpu + y  # Will fail if device != cpu
except RuntimeError as e:
    print(f"\nDevice mismatch error: {e}")

In [ ]:
# --- NumPy Interop: The Shared Memory Gotcha ---

# torch.from_numpy() shares memory with the NumPy array (zero-copy)
np_arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(np_arr)
print(f"NumPy: {np_arr}")
print(f"Tensor: {t}")

# Mutating one mutates the other!
np_arr[0] = 999.0
print(f"\nAfter modifying NumPy array:")
print(f"NumPy:  {np_arr}")
print(f"Tensor: {t}  <-- also changed!")

# Safe conversion: use .clone() to break shared memory
np_arr2 = np.array([10.0, 20.0, 30.0])
t_safe = torch.from_numpy(np_arr2).clone()
np_arr2[0] = -1.0
print(f"\nWith .clone():")
print(f"NumPy:  {np_arr2}")
print(f"Tensor: {t_safe}  <-- independent")

# .numpy() only works on CPU tensors
cpu_tensor = torch.randn(3)
as_numpy = cpu_tensor.numpy()  # shared memory again
print(f"\nTensor to NumPy: {as_numpy}")

In [ ]:
# --- Factory Functions: *_like ---
# These match shape, dtype, AND device of the input tensor.
# Critical in diffusion: you need noise on the same device as your image.

image_batch = torch.randn(8, 3, 64, 64, device=device)  # (B, C, H, W)

# These all inherit shape=(8,3,64,64), dtype=float32, device from image_batch
noise = torch.randn_like(image_batch)   # (B, C, H, W) -- sample noise for diffusion
zeros = torch.zeros_like(image_batch)   # (B, C, H, W) -- e.g., accumulator
ones = torch.ones_like(image_batch)     # (B, C, H, W) -- e.g., mask

print(f"image_batch: shape={image_batch.shape}, device={image_batch.device}")
print(f"noise:       shape={noise.shape}, device={noise.device}")
print(f"zeros:       shape={zeros.shape}, device={zeros.device}")

# In diffusion forward process: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
# noise = torch.randn_like(x_0)  <-- this is how it's done

### Exercise 1.1: Device Utility and Diffusion Noise Sampling

**1. `get_device()`**

Return the best available device: MPS > CUDA > CPU.

**2. `sample_noise(x)`**

Return Gaussian noise matching the input tensor's shape, dtype, and device.

- This is exactly the noise sampling step in the diffusion forward process
- Every training iteration calls a function like this to sample $\varepsilon \sim \mathcal{N}(0, I)$

In [ ]:
# YOUR CODE HERE

def get_device() -> torch.device:
    """Return the best available device: MPS > CUDA > CPU."""
    # ===================== YOUR CODE HERE =====================
    pass  # Replace this with your implementation
    # ====================== END YOUR CODE ======================


def sample_noise(x: torch.Tensor) -> torch.Tensor:
    """Sample Gaussian noise matching x's shape, dtype, and device.
    
    In diffusion models, this is called every training step to sample
    epsilon ~ N(0, I) with the same shape as the clean image x_0.
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace this with your implementation
    # ====================== END YOUR CODE ======================


# Tests
dev = get_device()
assert isinstance(dev, torch.device)
print(f"Best device: {dev}")

test_img = torch.randn(4, 3, 32, 32, device=dev)  # (B, C, H, W)
eps = sample_noise(test_img)
assert eps.shape == test_img.shape, f"Shape mismatch: {eps.shape} vs {test_img.shape}"
assert eps.device == test_img.device, f"Device mismatch: {eps.device} vs {test_img.device}"
assert eps.dtype == test_img.dtype, f"Dtype mismatch: {eps.dtype} vs {test_img.dtype}"
# Noise should be approximately N(0,1)
assert abs(eps.mean().item()) < 0.1, "Mean should be close to 0"
assert abs(eps.std().item() - 1.0) < 0.1, "Std should be close to 1"
print(f"Noise: shape={eps.shape}, mean={eps.mean():.4f}, std={eps.std():.4f}")
print("All tests passed.")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def get_device() -> torch.device:
    """Return the best available device: MPS > CUDA > CPU."""
    if torch.backends.mps.is_available():
        return torch.device('mps')
    elif torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')


def sample_noise(x: torch.Tensor) -> torch.Tensor:
    """Sample Gaussian noise matching x's shape, dtype, and device.
    
    In diffusion models, this is called every training step to sample
    epsilon ~ N(0, I) with the same shape as the clean image x_0.
    """
    return torch.randn_like(x)


# Tests
dev = get_device()
assert isinstance(dev, torch.device)
print(f"Best device: {dev}")

test_img = torch.randn(4, 3, 32, 32, device=dev)  # (B, C, H, W)
eps = sample_noise(test_img)
assert eps.shape == test_img.shape, f"Shape mismatch: {eps.shape} vs {test_img.shape}"
assert eps.device == test_img.device, f"Device mismatch: {eps.device} vs {test_img.device}"
assert eps.dtype == test_img.dtype, f"Dtype mismatch: {eps.dtype} vs {test_img.dtype}"
assert abs(eps.mean().item()) < 0.1, "Mean should be close to 0"
assert abs(eps.std().item() - 1.0) < 0.1, "Std should be close to 1"
print(f"Noise: shape={eps.shape}, mean={eps.mean():.4f}, std={eps.std():.4f}")
print("All tests passed.")

---
## 1.2 — Autograd Deep Dive

PyTorch builds a **dynamic computation graph** during the forward pass. Every operation on a tensor with `requires_grad=True` records itself in this graph. Calling `.backward()` traverses the graph in reverse to compute gradients via the chain rule.

### Dynamic graphs

The graph is rebuilt from scratch on every forward pass (unlike TensorFlow 1.x static graphs). This means you can use Python control flow — `if/else`, loops, even recursion — and the graph adapts automatically.

### Eager execution

Operations execute immediately, so you can `print()` intermediate values while debugging. No "session.run()" ceremony.

### The gradient accumulation gotcha

Gradients **accumulate** in `.grad` — they add up across calls to `.backward()`. If you forget to zero them between optimizer steps, your updates will be wrong. This is the single most common PyTorch bug for beginners.

In [ ]:
# --- Basic Autograd ---

# Create a tensor that tracks gradients
x = torch.tensor([2.0, 3.0], requires_grad=True)
print(f"x = {x}")
print(f"x.requires_grad = {x.requires_grad}")
print(f"x.grad_fn = {x.grad_fn}")  # None -- it's a leaf tensor

# Forward pass: y = x^2 + 3x
y = x ** 2 + 3 * x  # (2,)
print(f"\ny = x^2 + 3x = {y}")
print(f"y.grad_fn = {y.grad_fn}")  # AddBackward0 -- tracks the operation

# Scalar loss (backward requires a scalar)
loss = y.sum()  # scalar
print(f"\nloss = y.sum() = {loss}")

# Backward pass: compute d(loss)/dx
loss.backward()

# Gradient: d(loss)/dx = d(x^2 + 3x)/dx = 2x + 3
print(f"\nx.grad = {x.grad}")  # [2*2+3, 2*3+3] = [7, 9]
print(f"Expected: {2 * x.detach() + 3}")

In [ ]:
# --- The Gradient Accumulation Gotcha ---
# Gradients ACCUMULATE in .grad. If you don't zero them, you get wrong results.

w = torch.tensor([1.0], requires_grad=True)

# Step 1
loss1 = (w * 2).sum()
loss1.backward()
print(f"After step 1: w.grad = {w.grad}")  # [2.0]

# Step 2 -- WITHOUT zeroing gradients
loss2 = (w * 3).sum()
loss2.backward()
print(f"After step 2 (no zero): w.grad = {w.grad}")  # [5.0] = 2.0 + 3.0  <-- BUG!

# The fix: zero gradients before each backward pass
w.grad.zero_()  # Reset
loss3 = (w * 3).sum()
loss3.backward()
print(f"After step 3 (with zero): w.grad = {w.grad}")  # [3.0]  <-- correct

# In training loops, this is done by optimizer.zero_grad()
# Forgetting it is one of the most common PyTorch bugs.

In [ ]:
# --- torch.no_grad() and detach() ---

# torch.no_grad(): disables gradient tracking inside the block.
# Saves memory during inference (no need to store intermediate values for backward).

x = torch.randn(1000, 1000, requires_grad=True)

# With gradient tracking (training)
y_train = x ** 2 + x  # Graph is built, intermediates stored
print(f"With grad:    y.requires_grad = {y_train.requires_grad}")

# Without gradient tracking (inference)
with torch.no_grad():
    y_eval = x ** 2 + x  # No graph, no intermediate storage
    print(f"In no_grad(): y.requires_grad = {y_eval.requires_grad}")

# detach(): creates a tensor that shares data but is detached from the graph.
# Useful when you want to use a computed value without backpropagating through it.
z = x ** 2  # z is in the graph
z_detached = z.detach()  # z_detached shares data but is NOT in the graph
print(f"\nz.requires_grad = {z.requires_grad}")
print(f"z_detached.requires_grad = {z_detached.requires_grad}")

# In diffusion: detach() is used when computing the target.
# The noise prediction network's output gets gradients,
# but the target noise (epsilon) must be detached.

In [ ]:
# --- Inspecting the Computation Graph via grad_fn ---

a = torch.tensor(2.0, requires_grad=True)
b = a * 3        # MulBackward0
c = b + 1        # AddBackward0
d = c ** 2       # PowBackward0

print(f"a.grad_fn = {a.grad_fn}")  # None (leaf)
print(f"b.grad_fn = {b.grad_fn}")  # MulBackward0
print(f"c.grad_fn = {c.grad_fn}")  # AddBackward0
print(f"d.grad_fn = {d.grad_fn}")  # PowBackward0

# Walk backward through the graph
print(f"\nd depends on: {d.grad_fn.next_functions}")
print(f"c depends on: {c.grad_fn.next_functions}")

### Exercise 1.2: Manual Gradient Verification and Gradient Clipping

**1. Gradient verification**

For $f(x, y) = x^2 y + y^3$, compute $\partial f/\partial x$ and $\partial f/\partial y$ analytically (by hand), then verify with autograd at $(x=2, y=3)$.

**2. `clip_gradients(parameters, max_norm)`**

Clip gradients by global norm. This is essential for stable training — large gradients cause exploding updates, especially early in training.

- Compute the global L2 norm across all parameter gradients
- If it exceeds `max_norm`, scale all gradients down proportionally

In [ ]:
# YOUR CODE HERE — Exercise 1.2

# Part 1: Manual gradient verification
# f(x, y) = x^2 * y + y^3
# df/dx = ???
# df/dy = ???
# Evaluate at x=2, y=3

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

# ===================== YOUR CODE HERE =====================
# Compute f, call backward, then store your manual calculations
f = None  # Replace: compute f(x, y)
# Call f.backward() here

manual_dx = None  # Replace: what is df/dx at (2, 3)?
manual_dy = None  # Replace: what is df/dy at (2, 3)?
# ====================== END YOUR CODE ======================


# Tests — run this cell to check Part 1
assert f is not None, "Compute f = x^2 * y + y^3"
assert x.grad is not None, "Call f.backward() to compute gradients"
assert torch.isclose(x.grad, torch.tensor(manual_dx)), f"df/dx: autograd={x.grad.item()}, manual={manual_dx} — check your derivative"
assert torch.isclose(y.grad, torch.tensor(manual_dy)), f"df/dy: autograd={y.grad.item()}, manual={manual_dy} — check your derivative"
print(f"f(2, 3) = {f.item()}")
print(f"df/dx: autograd={x.grad.item()}, manual={manual_dx}")
print(f"df/dy: autograd={y.grad.item()}, manual={manual_dy}")
print("Part 1 ✓\n")


# Part 2: Gradient clipping
def clip_gradients(parameters: List[torch.Tensor], max_norm: float) -> float:
    """Clip gradients by global norm.
    
    Computes the global L2 norm across all parameter gradients.
    If it exceeds max_norm, scales all gradients down proportionally.
    
    Args:
        parameters: list of tensors with .grad attributes
        max_norm: maximum allowed global norm
    
    Returns:
        The original global norm (before clipping)
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace this with your implementation
    # ====================== END YOUR CODE ======================


# Tests — run this cell to check Part 2
torch.manual_seed(42)
_w1 = torch.randn(10, 10, requires_grad=True)
_w2 = torch.randn(10, 5, requires_grad=True)
_loss = (_w1.sum() * 100 + _w2.sum() * 50)
_loss.backward()
_norm_before = clip_gradients([_w1, _w2], max_norm=1.0)
_norm_after = math.sqrt(sum(p.grad.data.norm(2).item() ** 2 for p in [_w1, _w2]))
assert _norm_before > 1.0, "Norm before clipping should be > 1.0 for this test"
assert _norm_after <= 1.0 + 1e-5, f"After clipping, norm should be <= 1.0 but got {_norm_after:.4f}"
print(f"Global norm before: {_norm_before:.4f}")
print(f"Global norm after:  {_norm_after:.4f}")
print("Part 2 ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# Part 1: Manual gradient verification
# f(x, y) = x^2 * y + y^3
# df/dx = 2xy
# df/dy = x^2 + 3y^2

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

f = x ** 2 * y + y ** 3  # scalar
f.backward()

# Manual: df/dx = 2*2*3 = 12, df/dy = 4 + 27 = 31
manual_dx = 2 * 2.0 * 3.0  # 12.0
manual_dy = 2.0 ** 2 + 3 * 3.0 ** 2  # 31.0

print(f"f(2, 3) = {f.item()}")
print(f"df/dx: autograd={x.grad.item()}, manual={manual_dx}")
print(f"df/dy: autograd={y.grad.item()}, manual={manual_dy}")
assert torch.isclose(x.grad, torch.tensor(manual_dx)), "df/dx mismatch"
assert torch.isclose(y.grad, torch.tensor(manual_dy)), "df/dy mismatch"
print("Part 1 passed.\n")


# Part 2: Gradient clipping
def clip_gradients(parameters: List[torch.Tensor], max_norm: float) -> float:
    """Clip gradients by global norm.
    
    Computes the global L2 norm across all parameter gradients.
    If it exceeds max_norm, scales all gradients down proportionally.
    
    Args:
        parameters: list of tensors with .grad attributes
        max_norm: maximum allowed global norm
    
    Returns:
        The original global norm (before clipping)
    """
    # Compute global norm: sqrt(sum of squared norms of each gradient)
    total_norm_sq = 0.0
    grads = []
    for p in parameters:
        if p.grad is not None:
            total_norm_sq += p.grad.data.norm(2).item() ** 2
            grads.append(p.grad.data)
    total_norm = math.sqrt(total_norm_sq)
    
    # Clip: scale all gradients by max_norm / total_norm
    clip_coef = max_norm / (total_norm + 1e-6)
    if clip_coef < 1.0:
        for g in grads:
            g.mul_(clip_coef)
    
    return total_norm


# Test gradient clipping
torch.manual_seed(42)
w1 = torch.randn(10, 10, requires_grad=True)
w2 = torch.randn(10, 5, requires_grad=True)

# Create large gradients
loss = (w1.sum() * 100 + w2.sum() * 50)
loss.backward()

params = [w1, w2]
norm_before = clip_gradients(params, max_norm=1.0)
norm_after = math.sqrt(sum(p.grad.data.norm(2).item() ** 2 for p in params))

print(f"Global norm before clipping: {norm_before:.4f}")
print(f"Global norm after clipping:  {norm_after:.4f}")
assert norm_after <= 1.0 + 1e-5, f"Norm should be <= 1.0, got {norm_after}"
print("Part 2 passed.")

---
## 1.3 — In-Place Operations and Why They Break Autograd

In-place operations (those ending in `_`, like `add_()`, `mul_()`, `relu_()`) modify a tensor's data directly instead of creating a new tensor. They're dangerous during training because they can overwrite values that the backward pass needs.

| Operation | In-place | Out-of-place |
|---|---|---|
| Addition | `x.add_(y)` | `x = x + y` |
| Multiplication | `x.mul_(y)` | `x = x * y` |
| ReLU | `F.relu_(x)` | `F.relu(x)` |
| Slice assign | `x[:, 0] = val` | `x = torch.cat(...)` |

**The rule:** never use in-place ops on tensors that are part of a computation graph. During inference (inside `torch.no_grad()`), they're fine and can save memory.

In [ ]:
# --- In-place op that causes a RuntimeError ---

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * 2  # (3,) -- y is in the graph, depends on x

# In-place modification of y AFTER it's been used to build the graph
try:
    y.add_(1)  # Modifies y's data in-place
    loss = y.sum()
    loss.backward()  # RuntimeError!
except RuntimeError as e:
    print(f"RuntimeError: {e}\n")

# The safe alternative: out-of-place
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * 2       # (3,)
y = y + 1        # Creates a NEW tensor -- safe!
loss = y.sum()   # scalar
loss.backward()
print(f"Safe gradient: x.grad = {x.grad}")  # [2, 2, 2]

In [ ]:
# --- When in-place IS safe ---

# 1. On leaf tensors before any operation uses them
w = torch.zeros(3, requires_grad=True)
# w.data.fill_(1.0)  # OK -- modifying .data directly (use cautiously)

# 2. Inside torch.no_grad() -- no graph is being built
x = torch.randn(3, requires_grad=True)
with torch.no_grad():
    x.mul_(0.99)  # Fine -- used for weight decay or EMA updates
print(f"After in-place mul in no_grad: x = {x}")

# 3. On tensors that are not part of any graph
buffer = torch.zeros(10)  # No requires_grad
buffer.add_(1)  # Fine -- not tracked
print(f"Buffer after in-place add: {buffer[:5]}")

### Exercise 1.3: Identify Safe vs Unsafe In-Place Operations

For each snippet below, predict what happens:

- Works correctly
- Silently gives wrong gradients
- Raises a `RuntimeError`

Write your prediction as a comment, then uncomment the code to check.

In [ ]:
# YOUR CODE HERE: For each snippet, write your prediction as a comment,
# then uncomment the code to verify.

# Snippet A
# Prediction: ???
# ========================= YOUR CODE HERE =========================

a = torch.tensor([1.0, 2.0], requires_grad=True)
b = a + 1
# b.relu_()  # In-place ReLU on non-leaf in graph
# loss_a = b.sum()
# loss_a.backward()
# print(f"A: a.grad = {a.grad}")

# Snippet B
# Prediction: ???
c = torch.tensor([1.0, -1.0], requires_grad=True)
# d = F.relu(c)  # Out-of-place -- creates new tensor
# loss_b = d.sum()
# loss_b.backward()
# print(f"B: c.grad = {c.grad}")

# Snippet C
# Prediction: ???
e = torch.randn(3, requires_grad=True)
# with torch.no_grad():
#     e.add_(0.1)  # In-place, but inside no_grad
# f = e * 2
# loss_c = f.sum()
# loss_c.backward()
# print(f"C: e.grad = {e.grad}")

# ========================== END YOUR CODE ==========================

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# Snippet A: RuntimeError -- b is a non-leaf tensor in the graph,
# in-place relu_ overwrites values needed for backward.
print("--- Snippet A ---")
a = torch.tensor([1.0, 2.0], requires_grad=True)
b = a + 1
try:
    b.relu_()  # In-place on graph tensor
    loss_a = b.sum()
    loss_a.backward()
    print(f"A: a.grad = {a.grad}")
except RuntimeError as e:
    print(f"A: RuntimeError (as expected): {str(e)[:80]}...")

# Snippet B: Works correctly -- F.relu is out-of-place.
# Gradient: d(relu(x))/dx = 1 if x > 0, else 0
print("\n--- Snippet B ---")
c = torch.tensor([1.0, -1.0], requires_grad=True)
d = F.relu(c)  # Out-of-place
loss_b = d.sum()
loss_b.backward()
print(f"B: c.grad = {c.grad}")  # [1.0, 0.0] -- correct

# Snippet C: Works but subtle -- the in-place add_ modifies e's data,
# but since it's inside no_grad(), the graph isn't affected.
# The subsequent e * 2 uses the modified value of e.
print("\n--- Snippet C ---")
e = torch.randn(3, requires_grad=True)
with torch.no_grad():
    e.add_(0.1)  # Modifies e's data, no graph impact
f_val = e * 2
loss_c = f_val.sum()
loss_c.backward()
print(f"C: e.grad = {e.grad}")  # [2.0, 2.0, 2.0] -- gradient of e*2 is 2

---
## 1.4 — nn.Module Anatomy

`nn.Module` is the base class for every neural network component in PyTorch. To build custom architectures, you need to understand the four things a module can hold:

### Parameters (`nn.Parameter`)

These are the learnable weights — what the optimizer updates. Assign them as `self.weight = nn.Parameter(...)` and they automatically appear in `model.parameters()`, get saved in `state_dict()`, and move when you call `.to(device)`.

### Buffers (`register_buffer`)

Persistent state that is **not** learnable. Buffers get saved and moved to GPU, but the optimizer ignores them. In diffusion models, you'll store the entire noise schedule ($\bar{\alpha}_t$ values) as buffers.

### Submodules

Assign another `nn.Module` as an attribute (e.g., `self.layer = nn.Linear(...)`) and its parameters are registered automatically. This is how you compose complex architectures.

### Plain tensors (usually a bug)

If you write `self.x = torch.randn(...)` without wrapping it in `nn.Parameter` or `register_buffer`, PyTorch can't see it. It won't be saved, won't move to GPU, and won't show up in `state_dict()`. This is almost always a mistake.

| Concept | In `parameters()` | In `state_dict()` | Moved by `.to()` |
|---|---|---|---|
| `nn.Parameter` | Yes | Yes | Yes |
| Buffer | No | Yes | Yes |
| Plain tensor | No | No | No |
| Submodule | Its params | Yes | Yes |

In [ ]:
# --- nn.Module Internals ---

class DemoModule(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        # Submodule -- its parameters are registered automatically
        self.linear = nn.Linear(in_features, out_features)  # weight + bias
        
        # Explicit parameter -- registered, in parameters(), saved, moved by .to()
        self.scale = nn.Parameter(torch.ones(out_features))  # (out_features,)
        
        # Buffer -- NOT a parameter, but saved and moved by .to()
        self.register_buffer('running_mean', torch.zeros(out_features))  # (out_features,)
        
        # Plain tensor -- NOT registered, NOT saved, NOT moved by .to()
        self.unregistered = torch.randn(out_features)  # (out_features,) -- INVISIBLE
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.linear(x)   # (B, out_features)
        out = out * self.scale  # (B, out_features)
        return out


model = DemoModule(10, 5)

# Parameters: what the optimizer sees
print("--- Parameters ---")
for name, param in model.named_parameters():
    print(f"  {name}: shape={param.shape}, requires_grad={param.requires_grad}")

# Buffers
print("\n--- Buffers ---")
for name, buf in model.named_buffers():
    print(f"  {name}: shape={buf.shape}, requires_grad={buf.requires_grad}")

# Total parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params}")

# .to() moves parameters AND buffers, but NOT plain tensors
model = model.to(device)
print(f"\nlinear.weight device: {model.linear.weight.device}")
print(f"running_mean device:  {model.running_mean.device}")
print(f"unregistered device:  {model.unregistered.device}")  # Still on CPU!

In [ ]:
# --- train() vs eval() ---

# model.train() and model.eval() change behavior of Dropout and BatchNorm.
# They set self.training = True/False recursively on all submodules.

class TrainEvalDemo(nn.Module):
    def __init__(self):
        super().__init__()
        self.dropout = nn.Dropout(p=0.5)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(x)  # (same shape as x)

demo = TrainEvalDemo()
x = torch.ones(1, 10)  # (1, 10)

demo.train()
print(f"Train mode: {demo(x)}")  # Some values zeroed out

demo.eval()
print(f"Eval mode:  {demo(x)}")  # All values preserved

In [ ]:
# --- ModuleList, ModuleDict, Sequential ---

# nn.Sequential: simple chain of layers
seq = nn.Sequential(
    nn.Linear(10, 20),   # (B, 10) -> (B, 20)
    nn.ReLU(),
    nn.Linear(20, 5),    # (B, 20) -> (B, 5)
)
print(f"Sequential output: {seq(torch.randn(2, 10)).shape}")  # (2, 5)

# nn.ModuleList: indexed collection (use when you need loop access)
layers = nn.ModuleList([nn.Linear(10, 10) for _ in range(3)])
x = torch.randn(2, 10)  # (2, 10)
for layer in layers:
    x = F.relu(layer(x))  # (2, 10)
print(f"ModuleList output: {x.shape}")  # (2, 10)

# nn.ModuleDict: named collection (use for conditional architectures)
blocks = nn.ModuleDict({
    'up': nn.Linear(10, 20),
    'down': nn.Linear(20, 10),
})
y = blocks['up'](torch.randn(2, 10))   # (2, 20)
z = blocks['down'](y)                   # (2, 10)
print(f"ModuleDict output: {z.shape}")  # (2, 10)

# All three properly register their children's parameters
print(f"\nModuleList params: {sum(p.numel() for p in layers.parameters())}")
print(f"ModuleDict params: {sum(p.numel() for p in blocks.parameters())}")

### Exercise 1.4: TimestepEmbedding Module

Build a `TimestepEmbedding` module that converts integer timesteps to sinusoidal embeddings — a core component of every diffusion model.

The sinusoidal embedding for timestep $t$ with dimension $d$ is:

$$
\text{PE}(t, 2i)   = \sin\!\left(\frac{t}{10000^{2i/d}}\right), \qquad
\text{PE}(t, 2i+1) = \cos\!\left(\frac{t}{10000^{2i/d}}\right)
$$

Each timestep integer maps to a $d$-dimensional vector of sines and cosines at different frequencies. Low frequencies change slowly across timesteps; high frequencies change rapidly — giving the model a rich representation of "how noisy is this input?"

**Requirements:**

- Accept a batch of integer timesteps: `(B,)` → `(B, embed_dim)`
- Store the frequency table as a **buffer** (not a parameter — it's not learned)
- `embed_dim` must be even

In [ ]:
# YOUR CODE HERE — Exercise 1.4

class TimestepEmbedding(nn.Module):
    """Sinusoidal timestep embedding, as used in DDPM.
    
    Converts integer timesteps to dense vector representations using
    fixed sinusoidal frequencies (not learned).
    """
    
    def __init__(self, embed_dim: int):
        super().__init__()
        # ===================== YOUR CODE HERE =====================
        pass  # Replace this with your implementation
        # ====================== END YOUR CODE ======================
    
    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        """Args:
            timesteps: (B,) integer timesteps
        Returns:
            (B, embed_dim) sinusoidal embeddings
        """
        # ===================== YOUR CODE HERE =====================
        pass  # Replace this with your implementation
        # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
_emb = TimestepEmbedding(embed_dim=64).to(device)

# Should have no learned parameters (frequencies are fixed)
_n_params = sum(1 for _ in _emb.named_parameters())
assert _n_params == 0, f"Expected 0 parameters but found {_n_params} — frequencies should be buffers, not parameters"

# Should have freq buffer
_buffers = dict(_emb.named_buffers())
assert 'freq' in _buffers, "Missing 'freq' buffer — did you use register_buffer?"
print(f"Parameters: {_n_params} (correct — no learned params)")
print(f"Buffers: {list(_buffers.keys())}")

# Forward pass
_t = torch.tensor([0, 50, 100, 999], device=device)
_out = _emb(_t)
assert _out is not None, "forward() returned None"
assert _out.shape == (4, 64), f"Expected shape (4, 64) but got {_out.shape} — check your broadcasting"
assert _out.device.type == device.type, f"Output on {_out.device} but expected {device}"

# Different timesteps should produce different embeddings
_dists = torch.cdist(_out.unsqueeze(0), _out.unsqueeze(0)).squeeze()
assert _dists[0, 1] > 0.1, "t=0 and t=50 have identical embeddings — something is wrong"
print(f"Output shape: {_out.shape}")
print(f"Distance t=0 vs t=50:  {_dists[0, 1]:.4f}")
print(f"Distance t=0 vs t=999: {_dists[0, 3]:.4f}")
print("TimestepEmbedding ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class TimestepEmbedding(nn.Module):
    """Sinusoidal timestep embedding, as used in DDPM.
    
    Converts integer timesteps to dense vector representations using
    fixed sinusoidal frequencies (not learned).
    """
    
    def __init__(self, embed_dim: int):
        super().__init__()
        assert embed_dim % 2 == 0, "embed_dim must be even"
        self.embed_dim = embed_dim
        
        # Precompute frequency table: 1 / 10000^(2i/d) for i in [0, d/2)
        half_dim = embed_dim // 2
        freq = torch.exp(-math.log(10000.0) * torch.arange(half_dim, dtype=torch.float32) / half_dim)  # (half_dim,)
        self.register_buffer('freq', freq)  # Not a parameter -- fixed constant
    
    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        """Args:
            timesteps: (B,) integer timesteps
        Returns:
            (B, embed_dim) sinusoidal embeddings
        """
        # timesteps: (B,) -> (B, 1) for broadcasting
        args = timesteps.float().unsqueeze(-1) * self.freq.unsqueeze(0)  # (B, half_dim)
        embedding = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, embed_dim)
        return embedding


# Tests
emb_module = TimestepEmbedding(embed_dim=64)

print("--- Parameters (should be empty -- no learned params) ---")
for name, p in emb_module.named_parameters():
    print(f"  {name}: {p.shape}")
print(f"  Count: {sum(1 for _ in emb_module.named_parameters())}")

print("\n--- Buffers (should contain freq) ---")
for name, b in emb_module.named_buffers():
    print(f"  {name}: shape={b.shape}, device={b.device}")

# Move to device
emb_module = emb_module.to(device)
print(f"\nAfter .to({device}):")
print(f"  freq device: {emb_module.freq.device}")

# Forward pass
t = torch.tensor([0, 50, 100, 999], device=device)  # (4,)
embeddings = emb_module(t)  # (4, 64)
print(f"\nInput shape:  {t.shape}")
print(f"Output shape: {embeddings.shape}")
assert embeddings.shape == (4, 64)
assert embeddings.device.type == device.type

# Different timesteps should have different embeddings
dists = torch.cdist(embeddings.unsqueeze(0), embeddings.unsqueeze(0)).squeeze()  # (4, 4)
print(f"\nPairwise distances between timestep embeddings:")
print(f"  t=0 vs t=50:   {dists[0, 1]:.4f}")
print(f"  t=0 vs t=999:  {dists[0, 3]:.4f}")
print(f"  t=50 vs t=100: {dists[1, 2]:.4f}")
print("All tests passed.")

---
## 1.5 — Custom Layers from Scratch

Now let's implement standard layers ourselves. The key insight: most layers are just matrix multiplies with specific data layouts. Once you see the math, the mystery disappears.

### Weight initialization

Getting initialization right is surprisingly important. Without it, activations either vanish to zero or explode to infinity as they pass through many layers.

- **Xavier (Glorot):** `std = sqrt(2 / (fan_in + fan_out))` — best for sigmoid/tanh activations
- **Kaiming (He):** `std = sqrt(2 / fan_in)` — best for ReLU activations (the default in PyTorch)

In [ ]:
# --- MyLinear: nn.Linear from scratch ---

class MyLinear(nn.Module):
    """Linear layer: y = xW^T + b.
    
    Equivalent to nn.Linear. Implements Kaiming uniform initialization
    (same as PyTorch's default).
    """
    
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # Weight: (out_features, in_features)
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        
        # Bias: (out_features,)
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.bias = None
        
        # Initialize (matching PyTorch's default: Kaiming uniform)
        self._reset_parameters()
    
    def _reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = self.in_features
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Args:
            x: (..., in_features)
        Returns:
            (..., out_features)
        """
        # y = xW^T + b
        out = x @ self.weight.t()  # (..., out_features)
        if self.bias is not None:
            out = out + self.bias   # (..., out_features)
        return out


# Verify against nn.Linear
torch.manual_seed(42)
my_layer = MyLinear(10, 5)

torch.manual_seed(42)
pt_layer = nn.Linear(10, 5)

x = torch.randn(3, 10)  # (3, 10)
my_out = my_layer(x)    # (3, 5)
pt_out = pt_layer(x)    # (3, 5)

print(f"MyLinear output:  {my_out[0, :3]}")
print(f"nn.Linear output: {pt_out[0, :3]}")
print(f"Max difference:   {(my_out - pt_out).abs().max():.2e}")
assert torch.allclose(my_out, pt_out, atol=1e-6), "Outputs don't match!"
print("Numerically equivalent.")

### Exercise 1.5: MyConv2d and MyGroupNorm

**1. `MyConv2d`**

Implement 2D convolution using `F.unfold` (im2col) + matrix multiplication — this is how convolutions are actually computed on hardware.

- `F.unfold` extracts sliding patches from `(B, C_in, H, W)` into columns of shape `(B, C_in * kH * kW, L)` where `L` is the number of output positions
- Convolution then becomes a single matmul with the weight matrix
- Support square kernels, stride=1, and configurable padding

**2. `MyGroupNorm`**

Implement Group Normalization from scratch. Diffusion models use GroupNorm (not BatchNorm) because it works with any batch size and is stable across the varied noise levels of diffusion training.

- Split `C` channels into `G` groups, normalize each group independently
- For each group: `y = (x - mean) / sqrt(var + eps) * gamma + beta`
- Input: `(B, C, H, W)` → reshape to `(B, G, C//G, H, W)` → compute mean/var over last 3 dims

In [ ]:
# YOUR CODE HERE — Exercise 1.5

class MyConv2d(nn.Module):
    """2D convolution using F.unfold (im2col) + matmul.
    
    Supports only square kernels, stride=1, and specified padding.
    """
    
    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: int, padding: int = 0, bias: bool = True):
        super().__init__()
        # ===================== YOUR CODE HERE =====================
        pass  # Replace this with your implementation
        # ====================== END YOUR CODE ======================
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Args:
            x: (B, C_in, H, W)
        Returns:
            (B, C_out, H_out, W_out)
        """
        # ===================== YOUR CODE HERE =====================
        pass  # Replace this with your implementation
        # ====================== END YOUR CODE ======================


class MyGroupNorm(nn.Module):
    """Group Normalization from scratch.
    
    Splits channels into groups, normalizes each group independently.
    """
    
    def __init__(self, num_groups: int, num_channels: int, eps: float = 1e-5):
        super().__init__()
        # ===================== YOUR CODE HERE =====================
        pass  # Replace this with your implementation
        # ====================== END YOUR CODE ======================
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Args:
            x: (B, C, H, W)
        Returns:
            (B, C, H, W)
        """
        # ===================== YOUR CODE HERE =====================
        pass  # Replace this with your implementation
        # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work

# --- MyConv2d ---
torch.manual_seed(42)
_my_conv = MyConv2d(3, 16, kernel_size=3, padding=1)
torch.manual_seed(42)
_pt_conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)
_x = torch.randn(2, 3, 8, 8)

_my_out = _my_conv(_x)
_pt_out = _pt_conv(_x)
assert _my_out is not None, "MyConv2d forward returned None"
assert _my_out.shape == (2, 16, 8, 8), f"Expected (2, 16, 8, 8) but got {_my_out.shape} — check H_out/W_out calculation"
assert torch.allclose(_my_out, _pt_out, atol=1e-5), f"Max diff: {(_my_out - _pt_out).abs().max():.2e} — check unfold + matmul logic"
print(f"MyConv2d: {_my_out.shape}, max diff from nn.Conv2d: {(_my_out - _pt_out).abs().max():.2e}")
print("MyConv2d ✓\n")

# --- MyGroupNorm ---
torch.manual_seed(42)
_my_gn = MyGroupNorm(num_groups=4, num_channels=16)
_pt_gn = nn.GroupNorm(num_groups=4, num_channels=16)
with torch.no_grad():
    _my_gn.weight.copy_(_pt_gn.weight)
    _my_gn.bias.copy_(_pt_gn.bias)
_x2 = torch.randn(2, 16, 8, 8)

_my_out2 = _my_gn(_x2)
_pt_out2 = _pt_gn(_x2)
assert _my_out2 is not None, "MyGroupNorm forward returned None"
assert _my_out2.shape == (2, 16, 8, 8), f"Expected (2, 16, 8, 8) but got {_my_out2.shape}"
assert torch.allclose(_my_out2, _pt_out2, atol=1e-5), f"Max diff: {(_my_out2 - _pt_out2).abs().max():.2e} — check mean/var dims"
print(f"MyGroupNorm: {_my_out2.shape}, max diff from nn.GroupNorm: {(_my_out2 - _pt_out2).abs().max():.2e}")
print("MyGroupNorm ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class MyConv2d(nn.Module):
    """2D convolution using F.unfold (im2col) + matmul.
    
    Supports only square kernels, stride=1, and specified padding.
    """
    
    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: int, padding: int = 0, bias: bool = True):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.padding = padding
        
        # Weight: (out_channels, in_channels * kernel_size * kernel_size)
        self.weight = nn.Parameter(
            torch.empty(out_channels, in_channels, kernel_size, kernel_size)
        )
        if bias:
            self.bias = nn.Parameter(torch.empty(out_channels))
        else:
            self.bias = None
        
        # Kaiming init
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = in_channels * kernel_size * kernel_size
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Args:
            x: (B, C_in, H, W)
        Returns:
            (B, C_out, H_out, W_out)
        """
        B, C_in, H, W = x.shape
        
        # im2col: extract patches as columns
        # unfold output: (B, C_in * k * k, L) where L = H_out * W_out
        x_unfold = F.unfold(x, kernel_size=self.kernel_size, padding=self.padding)  # (B, C_in*k*k, L)
        
        # Reshape weight to (out_channels, C_in * k * k)
        w = self.weight.view(self.out_channels, -1)  # (C_out, C_in*k*k)
        
        # Matrix multiply: (C_out, C_in*k*k) @ (B, C_in*k*k, L) -> (B, C_out, L)
        out = torch.einsum('oi,bil->bol', w, x_unfold)  # (B, C_out, L)
        
        if self.bias is not None:
            out = out + self.bias.view(1, -1, 1)  # (B, C_out, L)
        
        # Reshape to spatial: (B, C_out, H_out, W_out)
        H_out = H + 2 * self.padding - self.kernel_size + 1
        W_out = W + 2 * self.padding - self.kernel_size + 1
        out = out.view(B, self.out_channels, H_out, W_out)  # (B, C_out, H_out, W_out)
        
        return out


class MyGroupNorm(nn.Module):
    """Group Normalization from scratch.
    
    Splits channels into groups, normalizes each group independently.
    """
    
    def __init__(self, num_groups: int, num_channels: int, eps: float = 1e-5):
        super().__init__()
        assert num_channels % num_groups == 0, "num_channels must be divisible by num_groups"
        self.num_groups = num_groups
        self.num_channels = num_channels
        self.eps = eps
        
        # Affine parameters: per-channel scale and shift
        self.weight = nn.Parameter(torch.ones(num_channels))   # gamma, (C,)
        self.bias = nn.Parameter(torch.zeros(num_channels))    # beta, (C,)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Args:
            x: (B, C, H, W)
        Returns:
            (B, C, H, W)
        """
        B, C, H, W = x.shape
        G = self.num_groups
        
        # Reshape: (B, G, C//G, H, W)
        x = x.view(B, G, C // G, H, W)
        
        # Compute mean and var per group (over channels-in-group, H, W)
        mean = x.mean(dim=(2, 3, 4), keepdim=True)  # (B, G, 1, 1, 1)
        var = x.var(dim=(2, 3, 4), keepdim=True, unbiased=False)  # (B, G, 1, 1, 1)
        
        # Normalize
        x = (x - mean) / torch.sqrt(var + self.eps)  # (B, G, C//G, H, W)
        
        # Reshape back: (B, C, H, W)
        x = x.view(B, C, H, W)
        
        # Apply affine: per-channel scale and shift
        x = x * self.weight.view(1, C, 1, 1) + self.bias.view(1, C, 1, 1)  # (B, C, H, W)
        
        return x


# --- Verify MyConv2d ---
torch.manual_seed(42)
my_conv = MyConv2d(3, 16, kernel_size=3, padding=1)
torch.manual_seed(42)
pt_conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)

x = torch.randn(2, 3, 8, 8)  # (B, C, H, W)
my_out = my_conv(x)   # (2, 16, 8, 8)
pt_out = pt_conv(x)   # (2, 16, 8, 8)

print(f"MyConv2d output shape: {my_out.shape}")
print(f"nn.Conv2d output shape: {pt_out.shape}")
print(f"Max difference: {(my_out - pt_out).abs().max():.2e}")
assert torch.allclose(my_out, pt_out, atol=1e-5), "Conv2d outputs don't match!"
print("MyConv2d verified.\n")

# --- Verify MyGroupNorm ---
torch.manual_seed(42)
my_gn = MyGroupNorm(num_groups=4, num_channels=16)
pt_gn = nn.GroupNorm(num_groups=4, num_channels=16)

# Copy weights to ensure same affine transform
with torch.no_grad():
    my_gn.weight.copy_(pt_gn.weight)
    my_gn.bias.copy_(pt_gn.bias)

x = torch.randn(2, 16, 8, 8)  # (B, C, H, W)
my_out = my_gn(x)   # (2, 16, 8, 8)
pt_out = pt_gn(x)   # (2, 16, 8, 8)

print(f"MyGroupNorm output shape: {my_out.shape}")
print(f"nn.GroupNorm output shape: {pt_out.shape}")
print(f"Max difference: {(my_out - pt_out).abs().max():.2e}")
assert torch.allclose(my_out, pt_out, atol=1e-5), "GroupNorm outputs don't match!"
print("MyGroupNorm verified.")

---
## 1.6 — Loss Functions from Raw Tensors

Let's build loss functions from scratch. In diffusion models, the training objective is remarkably simple:

$$
L = \mathbb{E}\left[\|\varepsilon - \varepsilon_\theta(x_t, t)\|^2\right]
$$

That's it — MSE between the true noise $\varepsilon$ and the predicted noise $\varepsilon_\theta(x_t, t)$. Predict what noise was added, penalize the squared error. That single line IS the diffusion training loss.

### Reduction modes

| Mode | Formula | When to use |
|---|---|---|
| `'mean'` | `loss.mean()` | Default — loss is independent of batch size |
| `'sum'` | `loss.sum()` | When you want gradients to scale with batch size |
| `'none'` | Element-wise | When you need per-sample losses (e.g., weighted by timestep) |

In [ ]:
# --- MSE Loss from Scratch ---

def my_mse_loss(pred: torch.Tensor, target: torch.Tensor,
                reduction: str = 'mean') -> torch.Tensor:
    """Mean Squared Error loss.
    
    Args:
        pred: predictions, any shape
        target: targets, same shape as pred
        reduction: 'mean', 'sum', or 'none'
    """
    squared_diff = (pred - target) ** 2  # element-wise
    if reduction == 'mean':
        return squared_diff.mean()
    elif reduction == 'sum':
        return squared_diff.sum()
    else:  # 'none'
        return squared_diff


# Verify against F.mse_loss
torch.manual_seed(42)
pred = torch.randn(4, 3, 32, 32)   # (B, C, H, W) -- noise prediction
target = torch.randn(4, 3, 32, 32) # (B, C, H, W) -- true noise

for reduction in ['mean', 'sum', 'none']:
    my_loss = my_mse_loss(pred, target, reduction)
    pt_loss = F.mse_loss(pred, target, reduction=reduction)
    if reduction == 'none':
        diff = (my_loss - pt_loss).abs().max()
    else:
        diff = (my_loss - pt_loss).abs()
    print(f"reduction='{reduction}': my={my_loss.mean():.6f}, pt={pt_loss.mean():.6f}, diff={diff:.2e}")
    assert torch.allclose(my_loss, pt_loss, atol=1e-6)

print("\nMSE loss verified.")
print("\nThis is the ENTIRE diffusion training loss:")
print("  loss = F.mse_loss(noise_pred, noise)")
print("  That's it. The simplicity is the point.")

In [ ]:
# --- Cross-Entropy Loss from Scratch ---
# Step by step: logits -> softmax -> log -> NLL

def my_cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Numerically stable cross-entropy loss.
    
    Uses the log-sum-exp trick to prevent overflow in softmax.
    
    Args:
        logits: (B, C) raw scores (NOT softmax outputs)
        targets: (B,) integer class labels
    Returns:
        scalar loss
    """
    B, C = logits.shape
    
    # Log-sum-exp trick for numerical stability:
    # log(sum(exp(x_i))) = max(x) + log(sum(exp(x_i - max(x))))
    max_logits = logits.max(dim=1, keepdim=True).values  # (B, 1)
    log_sum_exp = max_logits.squeeze(1) + torch.log(
        torch.exp(logits - max_logits).sum(dim=1)
    )  # (B,)
    
    # Log softmax = logits - log_sum_exp
    # NLL: pick the log-probability of the correct class
    correct_logits = logits[torch.arange(B), targets]  # (B,)
    
    # Loss = -log(softmax(logit_correct)) = log_sum_exp - correct_logit
    loss = log_sum_exp - correct_logits  # (B,)
    
    return loss.mean()  # scalar


# Verify
torch.manual_seed(42)
logits = torch.randn(8, 10)  # (B=8, num_classes=10)
targets = torch.randint(0, 10, (8,))  # (B=8,)

my_loss = my_cross_entropy(logits, targets)
pt_loss = F.cross_entropy(logits, targets)

print(f"My cross-entropy:     {my_loss:.6f}")
print(f"PyTorch cross-entropy: {pt_loss:.6f}")
print(f"Difference: {abs(my_loss - pt_loss):.2e}")
assert torch.allclose(my_loss, pt_loss, atol=1e-5)
print("Cross-entropy verified.")

### Exercise 1.6: Huber Loss and Weighted MSE

**1. Huber loss** (smooth L1)

Quadratic for small errors, linear for large errors — more robust to outliers than MSE.

$$
L_\delta(x) = \begin{cases} \frac{1}{2} x^2 & \text{if } |x| \leq \delta \\ \delta\,(|x| - \frac{1}{2}\delta) & \text{otherwise} \end{cases}
$$

- Behaves like MSE when the error is small (gradients proportional to error)
- Switches to linear penalty for large errors (outliers don't dominate)

**2. Weighted MSE**

In diffusion, some works weight the loss differently at different timesteps. Implement MSE where each sample gets a different weight.

- Weights shape: `(B,)` — predictions/targets shape: `(B, C, H, W)`
- You'll need to broadcast the weights correctly

In [ ]:
# YOUR CODE HERE — Exercise 1.6

def my_huber_loss(pred: torch.Tensor, target: torch.Tensor,
                  delta: float = 1.0, reduction: str = 'mean') -> torch.Tensor:
    """Huber (smooth L1) loss.
    
    Args:
        pred: predictions, any shape
        target: targets, same shape as pred
        delta: threshold for switching between quadratic and linear
        reduction: 'mean', 'sum', or 'none'
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace this with your implementation
    # ====================== END YOUR CODE ======================


def weighted_mse_loss(pred: torch.Tensor, target: torch.Tensor,
                      weights: torch.Tensor) -> torch.Tensor:
    """MSE loss with per-sample weights.
    
    In diffusion: different timesteps contribute differently to the loss.
    
    Args:
        pred: (B, ...) predictions
        target: (B, ...) targets
        weights: (B,) per-sample weights
    Returns:
        scalar weighted mean loss
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace this with your implementation
    # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
torch.manual_seed(42)
_pred = torch.randn(16)
_target = torch.randn(16)

for _reduction in ['mean', 'sum', 'none']:
    _my = my_huber_loss(_pred, _target, delta=1.0, reduction=_reduction)
    _pt = F.smooth_l1_loss(_pred, _target, reduction=_reduction, beta=1.0)
    assert _my is not None, f"my_huber_loss returned None for reduction='{_reduction}'"
    _diff = (_my - _pt).abs().max() if _reduction == 'none' else (_my - _pt).abs()
    assert _diff < 1e-6, f"Huber mismatch for reduction='{_reduction}': diff={_diff:.2e}"
    print(f"Huber reduction='{_reduction}': diff={_diff:.2e}")
print("Huber loss ✓\n")

# Weighted MSE: uniform weights should equal standard MSE
torch.manual_seed(42)
_pred2 = torch.randn(4, 3, 8, 8)
_target2 = torch.randn(4, 3, 8, 8)
_uniform_w = torch.ones(4)
_w_loss = weighted_mse_loss(_pred2, _target2, _uniform_w)
_mse = F.mse_loss(_pred2, _target2)
assert _w_loss is not None, "weighted_mse_loss returned None"
assert torch.allclose(_w_loss, _mse, atol=1e-5), f"Uniform weights should match standard MSE — got {_w_loss:.6f} vs {_mse:.6f}"
print(f"Weighted MSE (uniform): {_w_loss:.6f} vs standard MSE: {_mse:.6f}")
print("Weighted MSE ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def my_huber_loss(pred: torch.Tensor, target: torch.Tensor,
                  delta: float = 1.0, reduction: str = 'mean') -> torch.Tensor:
    """Huber (smooth L1) loss.
    
    Args:
        pred: predictions, any shape
        target: targets, same shape as pred
        delta: threshold for switching between quadratic and linear
        reduction: 'mean', 'sum', or 'none'
    """
    diff = pred - target
    abs_diff = diff.abs()
    
    # Quadratic regime: 0.5 * x^2
    quadratic = 0.5 * diff ** 2
    # Linear regime: delta * (|x| - 0.5 * delta)
    linear = delta * (abs_diff - 0.5 * delta)
    
    loss = torch.where(abs_diff <= delta, quadratic, linear)
    
    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    return loss


def weighted_mse_loss(pred: torch.Tensor, target: torch.Tensor,
                      weights: torch.Tensor) -> torch.Tensor:
    """MSE loss with per-sample weights.
    
    In diffusion: different timesteps contribute differently to the loss.
    E.g., early timesteps (low noise) might be weighted higher.
    
    Args:
        pred: (B, ...) predictions
        target: (B, ...) targets
        weights: (B,) per-sample weights
    Returns:
        scalar weighted mean loss
    """
    # Per-element squared difference
    se = (pred - target) ** 2  # (B, ...)
    
    # Mean over all dims except batch -> per-sample MSE
    # Flatten non-batch dims, then mean
    per_sample_mse = se.view(se.shape[0], -1).mean(dim=1)  # (B,)
    
    # Weighted mean
    return (weights * per_sample_mse).sum() / weights.sum()  # scalar


# --- Verify Huber Loss ---
torch.manual_seed(42)
pred = torch.randn(16)
target = torch.randn(16)

for reduction in ['mean', 'sum', 'none']:
    my_loss = my_huber_loss(pred, target, delta=1.0, reduction=reduction)
    pt_loss = F.smooth_l1_loss(pred, target, reduction=reduction, beta=1.0)
    if reduction == 'none':
        diff = (my_loss - pt_loss).abs().max()
    else:
        diff = (my_loss - pt_loss).abs()
    print(f"Huber reduction='{reduction}': diff={diff:.2e}")
    assert torch.allclose(my_loss, pt_loss, atol=1e-6)
print("Huber loss verified.\n")

# --- Test Weighted MSE ---
torch.manual_seed(42)
pred = torch.randn(4, 3, 8, 8)    # (B, C, H, W)
target = torch.randn(4, 3, 8, 8)  # (B, C, H, W)

# Uniform weights should equal normal MSE
uniform_w = torch.ones(4)  # (B,)
w_loss = weighted_mse_loss(pred, target, uniform_w)
mse_loss = F.mse_loss(pred, target)
print(f"Weighted MSE (uniform): {w_loss:.6f}")
print(f"Standard MSE:           {mse_loss:.6f}")
assert torch.allclose(w_loss, mse_loss, atol=1e-5), "Uniform weights should match standard MSE"

# Non-uniform weights: emphasize first sample
heavy_w = torch.tensor([10.0, 1.0, 1.0, 1.0])  # (B,)
w_loss_heavy = weighted_mse_loss(pred, target, heavy_w)
print(f"Weighted MSE (heavy):   {w_loss_heavy:.6f}")
print("Weighted MSE verified.")

---
## 1.7 — Optimizers: What They Actually Do

Optimizers update parameters based on gradients. Let's look at the update rules — understanding them matters when debugging training issues.

### SGD

The simplest optimizer: `theta -= lr * grad`. Noisy and slow to converge, but sometimes that's all you need.

### SGD + Momentum

Maintains a velocity `v = beta*v + grad`, then updates `theta -= lr*v`. The velocity smooths out noisy gradient directions, leading to faster convergence.

### Adam

Maintains two exponential moving averages per parameter: first moment $m$ (mean of gradients) and second moment $v$ (mean of squared gradients). This gives each parameter its own adaptive learning rate. Fast convergence, robust to hyperparameters.

### AdamW

Adam with **decoupled weight decay** — the weight regularization is applied directly to the parameters rather than through the gradient. This is the standard optimizer for diffusion models.

**Why Adam for diffusion?** Diffusion models train for many steps (often 500K+) with varied loss landscapes across timesteps. Adam's adaptive learning rates handle this naturally. Most papers use AdamW with `lr` in the range `1e-4` to `2e-4`.

In [ ]:
# --- Manual SGD Update ---

torch.manual_seed(42)
w = torch.randn(3, requires_grad=True)  # (3,)
lr = 0.1

# Forward + backward
loss = (w ** 2).sum()  # scalar
loss.backward()

print(f"Before: w = {w.data}")
print(f"Grad:   w.grad = {w.grad}")

# Manual SGD step: theta = theta - lr * grad
with torch.no_grad():
    w -= lr * w.grad

print(f"After:  w = {w.data}")

# Verify against torch.optim.SGD
torch.manual_seed(42)
w2 = torch.randn(3, requires_grad=True)
opt = torch.optim.SGD([w2], lr=0.1)

loss2 = (w2 ** 2).sum()
loss2.backward()
opt.step()

print(f"\nManual:  {w.data}")
print(f"Optim:   {w2.data}")
assert torch.allclose(w.data, w2.data, atol=1e-6)
print("Manual SGD matches torch.optim.SGD.")

In [ ]:
# --- Manual Adam Update ---
# Adam maintains two exponential moving averages per parameter:
#   m (first moment  -- mean of gradients)
#   v (second moment -- mean of squared gradients)
#
# Update at step t:
#   m = beta1 * m + (1 - beta1) * grad
#   v = beta2 * v + (1 - beta2) * grad^2
#   m_hat = m / (1 - beta1^t)     # bias correction
#   v_hat = v / (1 - beta2^t)     # bias correction
#   theta -= lr * m_hat / (sqrt(v_hat) + eps)

torch.manual_seed(42)
w = torch.randn(5, requires_grad=True)

# Adam hyperparameters
lr = 0.001
beta1, beta2 = 0.9, 0.999
eps = 1e-8

# Initialize moments
m = torch.zeros_like(w.data)  # (5,)
v = torch.zeros_like(w.data)  # (5,)

# Simulate 3 Adam steps
for step in range(1, 4):
    # Forward + backward
    if w.grad is not None:
        w.grad.zero_()
    loss = (w ** 2).sum()
    loss.backward()
    
    grad = w.grad.data
    
    # Update moments
    m = beta1 * m + (1 - beta1) * grad       # first moment
    v = beta2 * v + (1 - beta2) * grad ** 2   # second moment
    
    # Bias correction
    m_hat = m / (1 - beta1 ** step)
    v_hat = v / (1 - beta2 ** step)
    
    # Update
    with torch.no_grad():
        w -= lr * m_hat / (torch.sqrt(v_hat) + eps)
    
    print(f"Step {step}: loss={loss.item():.4f}, w[:3]={w.data[:3]}")

# Compare with torch.optim.Adam
torch.manual_seed(42)
w2 = torch.randn(5, requires_grad=True)
opt = torch.optim.Adam([w2], lr=lr, betas=(beta1, beta2), eps=eps)

for step in range(1, 4):
    opt.zero_grad()
    loss = (w2 ** 2).sum()
    loss.backward()
    opt.step()

print(f"\nManual Adam: w = {w.data[:3]}")
print(f"torch Adam:  w = {w2.data[:3]}")
print(f"Max diff: {(w.data - w2.data).abs().max():.2e}")
assert torch.allclose(w.data, w2.data, atol=1e-6)
print("Manual Adam matches torch.optim.Adam.")

In [ ]:
# --- SGD vs Adam: Training Curve Comparison ---

import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')

def train_with_optimizer(opt_class, opt_kwargs, n_steps=200):
    """Train a small network on a toy problem, return loss curve."""
    torch.manual_seed(0)
    # Toy problem: fit y = sin(x)
    x = torch.linspace(-3, 3, 100).unsqueeze(1)  # (100, 1)
    y = torch.sin(x)  # (100, 1)
    
    model = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 1))
    optimizer = opt_class(model.parameters(), **opt_kwargs)
    
    losses = []
    for _ in range(n_steps):
        optimizer.zero_grad()
        pred = model(x)  # (100, 1)
        loss = F.mse_loss(pred, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

sgd_losses = train_with_optimizer(torch.optim.SGD, {'lr': 0.01})
adam_losses = train_with_optimizer(torch.optim.Adam, {'lr': 0.01})
adamw_losses = train_with_optimizer(torch.optim.AdamW, {'lr': 0.01})

plt.figure(figsize=(8, 4))
plt.plot(sgd_losses, label='SGD (lr=0.01)', alpha=0.8)
plt.plot(adam_losses, label='Adam (lr=0.01)', alpha=0.8)
plt.plot(adamw_losses, label='AdamW (lr=0.01)', alpha=0.8)
plt.xlabel('Step')
plt.ylabel('MSE Loss')
plt.title('Optimizer Comparison: Fitting y = sin(x)')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise 1.7: Minimal Adam Optimizer from Scratch

Implement a complete `MyAdam` optimizer class.

**Requirements:**

- Track **first moment** ($m$), **second moment** ($v$), and step count per parameter
- Apply bias correction: $\hat{m} = m / (1 - \beta_1^t)$, $\hat{v} = v / (1 - \beta_2^t)$
- Implement `zero_grad()` and `step()`
- Train a small network and verify it matches `torch.optim.Adam`

In [ ]:
# YOUR CODE HERE — Exercise 1.7

class MyAdam:
    """Minimal Adam optimizer implementation.
    
    Follows the algorithm from Kingma & Ba, 2014.
    """
    
    def __init__(self, params: List[torch.Tensor], lr: float = 1e-3,
                 betas: Tuple[float, float] = (0.9, 0.999), eps: float = 1e-8):
        # ===================== YOUR CODE HERE =====================
        pass  # Replace this with your implementation
        # ====================== END YOUR CODE ======================
    
    def zero_grad(self):
        """Zero all parameter gradients."""
        # ===================== YOUR CODE HERE =====================
        pass  # Replace this with your implementation
        # ====================== END YOUR CODE ======================
    
    def step(self):
        """Perform one optimization step."""
        # ===================== YOUR CODE HERE =====================
        pass  # Replace this with your implementation
        # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
def _make_model_and_data():
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(5, 10), nn.ReLU(), nn.Linear(10, 1))
    x = torch.randn(20, 5)
    y = torch.randn(20, 1)
    return model, x, y

# Train with MyAdam
_m1, _x, _y = _make_model_and_data()
_my_opt = MyAdam(_m1.parameters(), lr=0.01)
for _ in range(50):
    _my_opt.zero_grad()
    _loss = F.mse_loss(_m1(_x), _y)
    _loss.backward()
    _my_opt.step()
_my_loss = F.mse_loss(_m1(_x), _y).item()

# Train with torch.optim.Adam
_m2, _x, _y = _make_model_and_data()
_pt_opt = torch.optim.Adam(_m2.parameters(), lr=0.01)
for _ in range(50):
    _pt_opt.zero_grad()
    _loss = F.mse_loss(_m2(_x), _y)
    _loss.backward()
    _pt_opt.step()
_pt_loss = F.mse_loss(_m2(_x), _y).item()

_max_diff = max(
    (p1.data - p2.data).abs().max().item()
    for p1, p2 in zip(_m1.parameters(), _m2.parameters())
)
print(f"MyAdam final loss:     {_my_loss:.6f}")
print(f"torch.Adam final loss: {_pt_loss:.6f}")
print(f"Max weight difference: {_max_diff:.2e}")
assert _max_diff < 1e-5, f"Weights diverged: max_diff={_max_diff:.2e} — check bias correction or moment updates"
print("MyAdam ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class MyAdam:
    """Minimal Adam optimizer implementation.
    
    Follows the algorithm from Kingma & Ba, 2014.
    """
    
    def __init__(self, params: List[torch.Tensor], lr: float = 1e-3,
                 betas: Tuple[float, float] = (0.9, 0.999), eps: float = 1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.t = 0  # step counter
        
        # Per-parameter state: first moment (m) and second moment (v)
        self.m = [torch.zeros_like(p.data) for p in self.params]
        self.v = [torch.zeros_like(p.data) for p in self.params]
    
    def zero_grad(self):
        """Zero all parameter gradients."""
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()
    
    def step(self):
        """Perform one optimization step."""
        self.t += 1
        
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            
            grad = p.grad.data
            
            # Update biased first moment estimate
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
            
            # Update biased second moment estimate
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * grad ** 2
            
            # Bias correction
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            
            # Update parameters
            p.data -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)


# --- Verify: Train same network with MyAdam and torch.optim.Adam ---

def make_model_and_data():
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(5, 10), nn.ReLU(), nn.Linear(10, 1))
    x = torch.randn(20, 5)   # (20, 5)
    y = torch.randn(20, 1)   # (20, 1)
    return model, x, y

# Train with MyAdam
model1, x, y = make_model_and_data()
my_opt = MyAdam(model1.parameters(), lr=0.01)

for step in range(50):
    my_opt.zero_grad()
    loss = F.mse_loss(model1(x), y)
    loss.backward()
    my_opt.step()
my_loss = F.mse_loss(model1(x), y).item()

# Train with torch.optim.Adam
model2, x, y = make_model_and_data()
pt_opt = torch.optim.Adam(model2.parameters(), lr=0.01)

for step in range(50):
    pt_opt.zero_grad()
    loss = F.mse_loss(model2(x), y)
    loss.backward()
    pt_opt.step()
pt_loss = F.mse_loss(model2(x), y).item()

print(f"MyAdam final loss:    {my_loss:.6f}")
print(f"torch.Adam final loss: {pt_loss:.6f}")

# Compare weights
max_diff = max(
    (p1.data - p2.data).abs().max().item()
    for p1, p2 in zip(model1.parameters(), model2.parameters())
)
print(f"Max weight difference: {max_diff:.2e}")
assert max_diff < 1e-5, f"Weights diverged: max_diff={max_diff}"
print("MyAdam matches torch.optim.Adam.")

---
## Capstone Exercise: MLP from Scratch on MNIST

Time to put everything together. Build a complete training pipeline with no high-level shortcuts:

- Use your `MyLinear` from Section 1.5 (no `nn.Linear`)
- Implement a custom `MyReLU` module (no `F.relu`)
- Use your cross-entropy loss from Section 1.6 (no `F.cross_entropy`)
- Write a manual training loop with `torch.optim.Adam`
- No `nn.Sequential` — wire everything manually in `forward()`

**Target: >95% accuracy on MNIST test set.**

If you hit that target, everything in this module is working — tensor creation, autograd, custom modules, custom layers, custom losses, and optimizer integration. That's every piece you need to start building diffusion models.

In [ ]:
# YOUR CODE HERE — Capstone Exercise

from torchvision import datasets, transforms
from torch.utils.data import DataLoader


class MyReLU(nn.Module):
    """ReLU activation from scratch."""
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ===================== YOUR CODE HERE =====================
        pass  # Replace this with your implementation
        # ====================== END YOUR CODE ======================


class CapstoneMLP(nn.Module):
    """MLP for MNIST using only custom layers.
    
    Architecture: 784 -> 256 -> ReLU -> 128 -> ReLU -> 10
    """
    def __init__(self):
        super().__init__()
        # Use MyLinear, MyReLU — no nn.Linear, no nn.Sequential
        # ===================== YOUR CODE HERE =====================
        pass  # Replace this with your implementation
        # ====================== END YOUR CODE ======================
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Args:
            x: (B, 1, 28, 28) MNIST images
        Returns:
            (B, 10) logits
        """
        # ===================== YOUR CODE HERE =====================
        pass  # Replace this with your implementation
        # ====================== END YOUR CODE ======================


def capstone_cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Cross-entropy loss from scratch (use your implementation from 1.6)."""
    # ===================== YOUR CODE HERE =====================
    pass  # Replace this with your implementation
    # ====================== END YOUR CODE ======================


def train_and_evaluate():
    """Train the MLP on MNIST and return test accuracy."""
    # ===================== YOUR CODE HERE =====================
    pass  # Replace this with your implementation
    # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
_model_check = CapstoneMLP()
assert _model_check is not None, "CapstoneMLP() returned None"
_n_params = sum(p.numel() for p in _model_check.parameters())
print(f"Model parameters: {_n_params:,}")

# Check no nn.Linear snuck in
for name, mod in _model_check.named_modules():
    assert not isinstance(mod, nn.Linear), f"Found nn.Linear at '{name}' — use MyLinear instead"
print("No nn.Linear found (using custom layers only) ✓")

# Quick forward pass test
_test_x = torch.randn(2, 1, 28, 28)
_test_out = _model_check(_test_x)
assert _test_out is not None, "forward() returned None"
assert _test_out.shape == (2, 10), f"Expected (2, 10) but got {_test_out.shape} — check flatten and layer dims"
print(f"Forward pass: (2, 1, 28, 28) → {_test_out.shape} ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

from torchvision import datasets, transforms
from torch.utils.data import DataLoader


class MyReLU(nn.Module):
    """ReLU activation from scratch."""
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.clamp(x, min=0)  # same shape as input


class CapstoneMLP(nn.Module):
    """MLP for MNIST using only custom layers.
    
    Architecture: 784 -> 256 -> ReLU -> 128 -> ReLU -> 10
    No nn.Linear, no nn.Sequential.
    """
    def __init__(self):
        super().__init__()
        self.fc1 = MyLinear(784, 256)    # (B, 784) -> (B, 256)
        self.act1 = MyReLU()
        self.fc2 = MyLinear(256, 128)    # (B, 256) -> (B, 128)
        self.act2 = MyReLU()
        self.fc3 = MyLinear(128, 10)     # (B, 128) -> (B, 10)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Args:
            x: (B, 1, 28, 28) MNIST images
        Returns:
            (B, 10) logits
        """
        x = x.view(x.shape[0], -1)   # (B, 784)
        x = self.act1(self.fc1(x))    # (B, 256)
        x = self.act2(self.fc2(x))    # (B, 128)
        x = self.fc3(x)               # (B, 10)
        return x


def capstone_cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Cross-entropy loss from scratch."""
    B, C = logits.shape
    max_logits = logits.max(dim=1, keepdim=True).values  # (B, 1)
    log_sum_exp = max_logits.squeeze(1) + torch.log(
        torch.exp(logits - max_logits).sum(dim=1)
    )  # (B,)
    correct_logits = logits[torch.arange(B, device=logits.device), targets]  # (B,)
    loss = (log_sum_exp - correct_logits).mean()  # scalar
    return loss


def train_and_evaluate():
    """Train the MLP on MNIST and return test accuracy."""
    # Data
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
    test_data = datasets.MNIST('./data', train=False, transform=transform)
    
    train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=256, shuffle=False)
    
    # Model and optimizer
    torch.manual_seed(42)
    model = CapstoneMLP()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Training
    n_epochs = 5
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0
        n_batches = 0
        
        for images, labels in train_loader:
            # Forward
            logits = model(images)  # (B, 10)
            loss = capstone_cross_entropy(logits, labels)  # scalar
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
        
        avg_loss = total_loss / n_batches
        
        # Evaluate
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                logits = model(images)  # (B, 10)
                preds = logits.argmax(dim=1)  # (B,)
                correct += (preds == labels).sum().item()
                total += labels.shape[0]
        
        accuracy = correct / total
        print(f"Epoch {epoch+1}/{n_epochs}: loss={avg_loss:.4f}, test_acc={accuracy:.4f}")
    
    print(f"\nFinal test accuracy: {accuracy:.4f}")
    assert accuracy > 0.95, f"Accuracy {accuracy:.4f} is below 95% target"
    print("Target accuracy (>95%) achieved.")
    return accuracy


# Verify parameter registration
model_check = CapstoneMLP()
n_params = sum(p.numel() for p in model_check.parameters())
print(f"Model parameters: {n_params:,}")
print(f"Layers: {[name for name, _ in model_check.named_modules() if name]}")
print()

In [ ]:
# Run the capstone training
accuracy = train_and_evaluate()
assert accuracy > 0.95, f"Accuracy {accuracy:.4f} is below 95% target"
print(f"\nFinal accuracy: {accuracy:.4f} — target achieved! ✓")

---
## Summary

Here's what you've built in this module — and how each piece connects to diffusion:

| Section | Key Takeaway | Diffusion Relevance |
|---|---|---|
| 1.1 Tensors | `randn_like(x)` matches shape/dtype/device | Noise sampling: `eps = torch.randn_like(x_0)` |
| 1.2 Autograd | Dynamic graph, gradient accumulation gotcha | Training loop: `loss.backward()`, `optimizer.zero_grad()` |
| 1.3 In-place | Avoid in-place ops in graph; safe in `no_grad()` | EMA update of model weights uses in-place under `no_grad()` |
| 1.4 nn.Module | Parameters, buffers, `.to(device)` | `register_buffer` for noise schedule ($\bar{\alpha}_t$ values) |
| 1.5 Custom Layers | Linear, Conv2d, GroupNorm from scratch | Diffusion U-Nets use Conv2d + GroupNorm extensively |
| 1.6 Loss Functions | MSE is the diffusion loss; log-sum-exp trick | `loss = F.mse_loss(noise_pred, noise)` |
| 1.7 Optimizers | Adam: adaptive LR, bias correction | AdamW with `lr=1e-4` is standard for diffusion training |

**Next up: Module 2 — Convolutions and Spatial Operations**, where we'll build the convolutional blocks that make up the U-Net architecture.